# Convert Gemini Model Predictions to Annotation Spans

This notebook ingests pre-annotated data from Gemini (TSV) and aligns label predictions with character spans. 

Then it formats them to fit the Label Studio format.

## Environment Setup & Imports

In [63]:
import pandas as pd
import ast
import re
import uuid
import json

In [64]:
# Paths
GEMINI_ANNOTATIONS_INPUT_PATH = "../data/input/gemini_pre_annotated_v3.tsv"
GEMINI_ANNOTATIONS_OUTPUT_PATH = "../data/output/pre-annotations-gemini_kys_2.json"
MODEL_VERSION = "gemini_pre_annotated_v9"

## Load Dataset

In [65]:
predictions = pd.read_csv(GEMINI_ANNOTATIONS_INPUT_PATH, sep='\t')

In [66]:
# Since metadata is stored inside a stringified dictionary within the `data` column, we extract the unique ID (note_id) and the raw string content (note_content).

predictions['note_id'] = predictions['data'].apply(lambda x: ast.literal_eval(x)['id'])
predictions['note_content'] = predictions['data'].apply(lambda x: ast.literal_eval(x)['note_content'])
predictions.head()

,data,predictions,gemini_predictions,note_id,note_content
0,{'note_content': '[Category: Rents] Telephone ...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ { ""text"": ""Telephone line just r...",4d1f84da-85f0-9837-33ac-bdc3ae96fcba,[Category: Rents] Telephone line just ringing
1,{'note_content': '[Category: tenureManagement]...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""tenant"", ""label"": ""Per...",685524bb-94d4-4f08-8669-0e78df744c0c,[Category: tenureManagement] Welfare check and...
2,{'note_content': '[Category: estateManagement]...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""tenant"", ""label"": ""Per...",b7b3fb41-80b7-4d97-b8f4-0e0a7156ab79,[Category: estateManagement] arson attack lett...
3,{'note_content': '[Category: repairs] Repair i...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""tenant"", ""label"": ""Per...",918721e6-ccf6-445c-8cfc-658826b245cd,[Category: repairs] Repair issue Sent further ...
4,{'note_content': '[Category: Tenancy Managemen...,"[{'model_version': 'pre_annotated', 'score': 1...","{ ""labels"": [ {""text"": ""Uju"", ""label"": ""Person...",5cac962d-dfc9-72d1-175a-612d7462cd4d,[Category: Tenancy Management] Dear Uju\n\nMs ...


## Data Validation

Verify that all generated values in the `gemini_predictions` column are valid JSON. 

Sometimes the model responds with something along the lines of "sorry, I can't do that." or some other invalid JSON, so this will catch that.

In [67]:
def is_valid_json(x):
    try:
        json.loads(x)
        return True
    except (ValueError, TypeError):
        return False

mask = predictions['gemini_predictions'].apply(lambda x: not is_valid_json(x))
valid_predictions = predictions[~mask]

display(predictions[mask])

,data,predictions,gemini_predictions,note_id,note_content


## Map Predictions to Character Offsets

Matches the raw predicted text segments to their character starting and ending indices using a sequential left-to-right regex scan. 

This helps when the same word/words are repeated in the note (e.g. 'tenant' may be mentioned multiple times). 

Without the left-to-right scan, the matching might highlight the same word multiple times instead of finding different instances.

In [68]:
def import_gemini_predictions(row):
    identified_labels = []
    try:
        gemini = json.loads(row.gemini_predictions)

        # Build span list and id_lookup together, in match order
        id_lookup = {}  # text.lower() -> [{"id": ..., "start": ..., "end": ...}, ...]

        for pred in gemini['labels']:
            text = pred['text']
            key = text.lower()
            already_found = len(id_lookup.get(key, []))
            all_matches = list(re.finditer(r'\b' + re.escape(text) + r'\b', row.note_content, re.IGNORECASE))

            for match in all_matches[already_found:]:
                entity_id = str(uuid.uuid4())[:8]
                is_entity = pred['label'] in ('Person_Name', 'Person_Role', 'Person_Pronoun')
                id_lookup.setdefault(key, []).append({"id": entity_id, "start": match.start(), "end": match.end()})
                identified_labels.append({
                    "id": entity_id,
                    "from_name": "entity_labels" if is_entity else "need_labels",
                    "to_name": "text",
                    "type": "labels",
                    "value": {
                        "start": match.start(),
                        "end": match.end(),
                        "text": match.group(),
                        "labels": [pred['label']]
                    }
                })

        def nearest_span(from_start, candidates):
            preceding = [c for c in candidates if c['start'] <= from_start]
            if preceding:
                return max(preceding, key=lambda c: c['start'])
            return min(candidates, key=lambda c: c['start'])

        for link in gemini['links']:
            from_key = link['from'].lower()
            to_key = link['to'].lower()

            from_candidates = id_lookup.get(from_key, [])
            to_candidates = id_lookup.get(to_key, [])

            if not from_candidates or not to_candidates:
                print(f"Warning: could not resolve link '{link['from']}' -> '{link['to']}' in {row.note_id}")
                continue

            from_span = nearest_span(len(row.note_content), from_candidates)  # from: take last occurrence
            
            # skip self-links
            if from_key == to_key:
                continue

            to_span = nearest_span(from_span['start'], to_candidates)

            identified_labels.append({
                "from_id": from_span['id'],
                "to_id": to_span['id'],
                "type": "relation",
                "direction": "right"
            })

    except Exception as e:
        print(f"Error parsing note id: {row.note_id}, {e}")

    return {
        "data": {
            "id": row.note_id,
            "note_content": row.note_content,
        },
        "predictions": [{
            "model_version": MODEL_VERSION,
            "score": 1.0,
            "result": identified_labels
        }]
    }

In [69]:
predictions_json = valid_predictions.apply(import_gemini_predictions, axis=1)

## Export as Label Studio JSON

In [70]:
with open(GEMINI_ANNOTATIONS_OUTPUT_PATH, 'w') as f:
    json.dump(predictions_json.tolist(), f, indent=4)